# Step 2 — Source Experimental Structures (Corrected)

**Pipeline:** Myocarditis structural mimicry reassessment
**Corrected version** incorporating all fixes from implementation review:
- Broadened candidate pool (TOP_N_SPIKE=60, secondary sort by polymer_entity_count)
- Canonical reference structures (6VXX, 6VYB, 6XRA) force-fetched and deterministically
  state-assigned, not dependent on fuzzy title classification
- Mixed/composite state detection prevents mislabeling (e.g. "1 RBD open + 2 RBD closed"
  no longer silently classified as "closed")

**Reads:** `panel.json` (Step 1)
**Produces:** `structure_sources.json` + downloaded .cif structure files

In [ ]:
import importlib, subprocess, sys

for pkg in ("requests",):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import requests
print("requests:", requests.__version__)

In [ ]:
import json, os

MOUNT_DRIVE = True
BASE_DIR = "/content"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/myocarditis_pipeline"

STEP1_DIR = os.path.join(BASE_DIR, "step1")
STEP2_DIR = os.path.join(BASE_DIR, "step2")
os.makedirs(STEP2_DIR, exist_ok=True)

panel_path = os.path.join(STEP1_DIR, "panel.json")
if not os.path.exists(panel_path):
    raise FileNotFoundError(f"panel.json not found at {panel_path}. Run Step 1 first.")

with open(panel_path) as f:
    panel = json.load(f)

print(f"Loaded panel.json (generated {panel['generated_at_utc']})")
for key, entry in panel["entries"].items():
    print(f"  {key}: {entry['accession']} ({entry['verified_length_aa']} aa)")

## RCSB Search API functions

In [ ]:
import time

RCSB_SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
RCSB_ENTRY_URL = "https://data.rcsb.org/rest/v1/core/entry"
RCSB_POLYMER_ENTITY_URL = "https://data.rcsb.org/rest/v1/core/polymer_entity"
REQUEST_TIMEOUT = 30


def rcsb_sequence_search(sequence, identity_cutoff=0.9, evalue_cutoff=1.0):
    query = {
        "query": {
            "type": "terminal",
            "service": "sequence",
            "parameters": {
                "evalue_cutoff": evalue_cutoff,
                "identity_cutoff": identity_cutoff,
                "sequence_type": "protein",
                "value": sequence,
            },
        },
        "return_type": "entry",
        "request_options": {
            "results_content_type": ["experimental"],
            "sort": [{"sort_by": "score", "direction": "desc"}],
            "return_all_hits": False,
        },
    }
    resp = requests.post(RCSB_SEARCH_URL, json=query, timeout=REQUEST_TIMEOUT)
    if resp.status_code == 204:
        return []
    resp.raise_for_status()
    data = resp.json()
    return [(hit["identifier"], hit.get("score")) for hit in data.get("result_set", [])]


def fetch_entry_metadata(pdb_id):
    resp = requests.get(f"{RCSB_ENTRY_URL}/{pdb_id}", timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    struct = data.get("struct", {})
    exptl = data.get("exptl", [{}])
    rcsb_info = data.get("rcsb_entry_info", {})
    resolution = rcsb_info.get("resolution_combined", [None])
    return {
        "pdb_id": pdb_id,
        "title": struct.get("title", ""),
        "method": exptl[0].get("method", "") if exptl else "",
        "resolution_A": resolution[0] if resolution else None,
        "release_date": data.get("rcsb_accession_info", {}).get("initial_release_date", ""),
        "polymer_entity_count": rcsb_info.get("polymer_entity_count_protein"),
    }


print("RCSB functions ready.")

## Spike: search with broadened pool and canonical cross-check

In [ ]:
spike_seq = panel["entries"]["spike"]["sequence"]
print(f"Spike sequence length: {len(spike_seq)} aa")
print("Querying RCSB by sequence similarity (experimental structures only)...\n")

spike_hits = rcsb_sequence_search(spike_seq, identity_cutoff=0.9)
print(f"{len(spike_hits)} experimental hits returned.\n")

TOP_N_SPIKE = 60
spike_candidates = []

for pdb_id, score in spike_hits[:TOP_N_SPIKE]:
    meta = fetch_entry_metadata(pdb_id)
    meta["search_score"] = score
    spike_candidates.append(meta)
    time.sleep(0.3)

print(f"{len(spike_candidates)} candidates retrieved.\n")

# Secondary sort: prefer simple spike-alone constructs
spike_candidates.sort(key=lambda c: (c.get("polymer_entity_count") or 99,
                                      c["resolution_A"] is None,
                                      c["resolution_A"] or 999))

for c in spike_candidates[:15]:
    print(f"  {c['pdb_id']}  entities={c['polymer_entity_count']}  "
          f"res={c['resolution_A']}A  \"{c['title'][:60]}\"")

# Canonical reference structures with deterministic state assignment
CANONICAL_REFERENCE_STATES = {
    "6VXX": ("closed", "canonical closed-state prefusion trimer (Walls et al. 2020)"),
    "6VYB": ("one_rbd_up", "canonical one-RBD-up prefusion trimer (Walls et al. 2020)"),
    "6XRA": ("postfusion", "full-length S2 trimer in postfusion conformation (Cai et al. 2020)"),
}

found_ids = {c["pdb_id"] for c in spike_candidates}
print("\nCanonical reference check (state assigned directly, not by classifier):")
for ref_id, (known_state, desc) in CANONICAL_REFERENCE_STATES.items():
    if ref_id in found_ids:
        print(f"  {ref_id} ({desc}): FOUND in pool")
    else:
        print(f"  {ref_id} ({desc}): MISSING -- fetching directly")
        try:
            meta = fetch_entry_metadata(ref_id)
            meta["search_score"] = None
            spike_candidates.append(meta)
            found_ids.add(ref_id)
            print(f"    -> added: \"{meta['title'][:60]}\"")
        except Exception as e:
            print(f"    -> WARNING: fetch failed: {e}")

In [ ]:
# Classify by conformational state with mixed-state detection
STATE_KEYWORDS = {
    "closed": ["all rbd down", "3-rbd-down", "3 rbd down", "tightly closed",
               "fully closed", "closed conformation", "closed state"],
    "one_rbd_up": ["one rbd up", "1 rbd up", "1-rbd-up", "single rbd up",
                   "single rbd open", "one rbd open", "one rbd in open state", "1 rbd open"],
    "two_rbd_up": ["two rbd up", "2 rbd up", "2-rbd-up", "double rbd up",
                   "two rbd open", "2 rbd open"],
    "postfusion": ["postfusion", "post-fusion", "post fusion"],
}

def classify_state(title):
    title_lower = title.lower()
    matches = [state for state, kws in STATE_KEYWORDS.items()
               if any(kw in title_lower for kw in kws)]
    if len(matches) == 1:
        return matches[0]
    elif len(matches) > 1:
        return "mixed_or_composite(" + ",".join(matches) + ")"
    return "unclassified"

for c in spike_candidates:
    c["classified_state"] = classify_state(c["title"])

# Apply canonical overrides AFTER classifier
n_overridden = 0
for c in spike_candidates:
    if c["pdb_id"] in CANONICAL_REFERENCE_STATES:
        known_state, _ = CANONICAL_REFERENCE_STATES[c["pdb_id"]]
        if c["classified_state"] != known_state:
            print(f"  Overriding {c['pdb_id']}: '{c['classified_state']}' -> '{known_state}'")
            n_overridden += 1
        c["classified_state"] = known_state

print(f"\nClassification results ({n_overridden} canonical override(s) applied):\n")
for c in spike_candidates:
    print(f"  {c['pdb_id']:6s}  {c['classified_state']:35s}  \"{c['title'][:60]}\"")

### Select final spike structures

In [ ]:
SPIKE_STATE_OVERRIDES = {}  # manual overrides if needed after reviewing output above

for pdb_id, forced_state in SPIKE_STATE_OVERRIDES.items():
    for c in spike_candidates:
        if c["pdb_id"] == pdb_id:
            c["classified_state"] = forced_state

TARGET_STATES = ["closed", "one_rbd_up", "two_rbd_up", "postfusion"]
selected_spike_structures = {}

for state in TARGET_STATES:
    matches = [c for c in spike_candidates if c["classified_state"] == state]
    if not matches:
        print(f"  {state}: NO MATCH -- documented gap")
        continue
    matches.sort(key=lambda c: (c["resolution_A"] is None, c["resolution_A"] or 999))
    best = matches[0]
    selected_spike_structures[state] = best
    print(f"  {state}: selected {best['pdb_id']} ({best['resolution_A']} A)")

print(f"\n{len(selected_spike_structures)} of {len(TARGET_STATES)} states covered.")

## Cardiac antigens: best experimental coverage

In [ ]:
CARDIAC_ANTIGENS = ["MYH6", "TPM1", "TPM3", "ACTC1"]
cardiac_results = {}

for symbol in CARDIAC_ANTIGENS:
    entry = panel["entries"][symbol]
    seq = entry["sequence"]
    print(f"\n{symbol} ({entry['accession']}, {len(seq)} aa) -- searching RCSB...")

    hits = rcsb_sequence_search(seq, identity_cutoff=0.9)
    print(f"  {len(hits)} hits found.")

    candidates = []
    for pdb_id, score in hits[:15]:
        meta = fetch_entry_metadata(pdb_id)
        meta["search_score"] = score
        candidates.append(meta)
        time.sleep(0.3)

    cardiac_results[symbol] = candidates
    if candidates:
        candidates.sort(key=lambda c: (c["resolution_A"] is None, c["resolution_A"] or 999))
        for c in candidates[:3]:
            print(f"    {c['pdb_id']}  res={c['resolution_A']}A  \"{c['title'][:55]}\"")
    else:
        print(f"    NO EXPERIMENTAL STRUCTURES FOUND")

In [ ]:
def fetch_best_coverage(pdb_id, target_length):
    resp = requests.get(f"{RCSB_ENTRY_URL}/{pdb_id}", timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    entry_data = resp.json()
    entity_ids = entry_data.get("rcsb_entry_container_identifiers", {}).get("polymer_entity_ids", [])

    best_entity = None
    for entity_id in entity_ids:
        e_resp = requests.get(f"{RCSB_POLYMER_ENTITY_URL}/{pdb_id}/{entity_id}", timeout=REQUEST_TIMEOUT)
        if e_resp.status_code != 200:
            continue
        e_data = e_resp.json()
        seq = e_data.get("entity_poly", {}).get("pdbx_seq_one_letter_code_can", "") or ""
        seq_len = len(seq.replace("\n", ""))
        if best_entity is None or seq_len > best_entity["seq_len"]:
            best_entity = {"entity_id": entity_id, "seq_len": seq_len}
        time.sleep(0.2)

    if best_entity is None:
        return {"covered_length": 0, "coverage_pct": 0.0, "entity_id": None}

    coverage_pct = round(100 * best_entity["seq_len"] / target_length, 1) if target_length else None
    return {"covered_length": best_entity["seq_len"], "coverage_pct": coverage_pct,
            "entity_id": best_entity["entity_id"]}


coverage_summary = {}
for symbol in CARDIAC_ANTIGENS:
    entry = panel["entries"][symbol]
    target_length = entry["verified_length_aa"]
    candidates = cardiac_results[symbol]

    if not candidates:
        coverage_summary[symbol] = {"best_pdb_id": None, "covered_length": 0, "coverage_pct": 0.0}
        continue

    candidates.sort(key=lambda c: (c["resolution_A"] is None, c["resolution_A"] or 999))
    best_pdb_id = candidates[0]["pdb_id"]
    cov = fetch_best_coverage(best_pdb_id, target_length)
    coverage_summary[symbol] = {"best_pdb_id": best_pdb_id, **cov, "target_length": target_length}

print("Coverage summary:\n")
for symbol, cov in coverage_summary.items():
    if cov["best_pdb_id"]:
        print(f"  {symbol}: {cov['best_pdb_id']} covers {cov['covered_length']}/{cov['target_length']} "
              f"aa ({cov['coverage_pct']}%)")
    else:
        print(f"  {symbol}: NO experimental coverage")

## Download selected structures

In [ ]:
STRUCTURES_DIR = os.path.join(STEP2_DIR, "structures")
os.makedirs(STRUCTURES_DIR, exist_ok=True)

def download_structure(pdb_id, dest_dir):
    url = f"https://files.rcsb.org/download/{pdb_id}.cif"
    resp = requests.get(url, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    dest_path = os.path.join(dest_dir, f"{pdb_id}.cif")
    with open(dest_path, "wb") as f:
        f.write(resp.content)
    return dest_path

# Reconstruct selected_spike_structures if needed
if not selected_spike_structures:
    for state in TARGET_STATES:
        matches = [c for c in spike_candidates if c["classified_state"] == state]
        if matches:
            matches.sort(key=lambda c: (c["resolution_A"] is None, c["resolution_A"] or 999))
            selected_spike_structures[state] = matches[0]

downloaded = {}

print("Downloading spike structures...")
for state, meta in selected_spike_structures.items():
    path = download_structure(meta["pdb_id"], STRUCTURES_DIR)
    downloaded[f"spike_{state}"] = {"pdb_id": meta["pdb_id"], "path": path}
    print(f"  {meta['pdb_id']} ({state}) -> {path}")
    time.sleep(0.3)

print("\nDownloading cardiac antigen structures...")
for symbol, cov in coverage_summary.items():
    if cov["best_pdb_id"]:
        path = download_structure(cov["best_pdb_id"], STRUCTURES_DIR)
        downloaded[symbol] = {"pdb_id": cov["best_pdb_id"], "path": path}
        print(f"  {symbol}: {cov['best_pdb_id']} -> {path}")
        time.sleep(0.3)
    else:
        downloaded[symbol] = None
        print(f"  {symbol}: nothing to download")

## Write `structure_sources.json`

In [ ]:
from datetime import datetime, timezone

structure_sources = {
    "pipeline_step": 2,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "generated_in": "Google Colab",
    "source_panel": panel_path,
    "spike": {
        "target_states": TARGET_STATES,
        "states_covered": list(selected_spike_structures.keys()),
        "states_missing": [s for s in TARGET_STATES if s not in selected_spike_structures],
        "selected": {
            state: {
                "pdb_id": meta["pdb_id"],
                "resolution_A": meta["resolution_A"],
                "method": meta["method"],
                "title": meta["title"],
                "release_date": meta["release_date"],
                "file_path": downloaded[f"spike_{state}"]["path"],
            }
            for state, meta in selected_spike_structures.items()
        },
        "all_candidates_considered": spike_candidates,
    },
    "cardiac_antigens": {
        symbol: {
            "target_length_aa": coverage_summary[symbol].get("target_length"),
            "best_pdb_id": coverage_summary[symbol]["best_pdb_id"],
            "covered_length_aa": coverage_summary[symbol]["covered_length"],
            "coverage_pct": coverage_summary[symbol]["coverage_pct"],
            "file_path": downloaded[symbol]["path"] if downloaded[symbol] else None,
            "needs_fallback": coverage_summary[symbol]["coverage_pct"] is None
                or coverage_summary[symbol]["coverage_pct"] < 100.0,
            "all_candidates_considered": cardiac_results[symbol],
        }
        for symbol in CARDIAC_ANTIGENS
    },
}

output_path = os.path.join(STEP2_DIR, "structure_sources.json")
with open(output_path, "w") as f:
    json.dump(structure_sources, f, indent=2)

print("structure_sources.json written to:", output_path)